# Data Cleaning Perfect - Terrain Location Marrakech
This notebook implements the 'Perfect Cleaning' pipeline for Machine Learning preparation.

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# File path
file_path = '../../data/marrakech_immo_location/terrain_location.csv'

# Load data
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"Successfully loaded {file_path}")
    print(f"Initial shape: {df.shape}")
    display(df.head())
else:
    print(f"ERROR: File not found at {file_path}")

Successfully loaded ../../data/marrakech_immo_location/terrain_location.csv
Initial shape: (18, 34)


,id,titre,prix,localisation,type_bien,surface,chambres,salles_bain,description,agence,...,etage,surface_terrain,prix_num,surface_num,chambres_num,salles_bain_num,nb_pieces,quartier,prix_m2,prix_m2_median_quartier
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1,NaN,NaN,NaN,0.0,0.0,1.0,Autre,NaN,NaN
1,ML-247,PROMO IMMO MARRAKECH VOUS PROPOSE LOCAL COMMER...,2000000 dhs,Av Moulay Abdallah,Terrain,330 m²,0.0,NaN,NaN,Promo Immo Marrakech,...,-1,NaN,2000000.0,330.0,0.0,0.0,1.0,Autre,6060.606061,6060.606061
2,TRL-01,terrain à louer,6670 dhs /Par mois,Mhamid,Terrain,m²,0.0,NaN,Agafay est un domaine d'exploitation touristiq...,Promo Immo Marrakech,...,-1,NaN,6670.0,NaN,0.0,0.0,1.0,M'hamid,NaN,NaN
3,NaN,Terrain agricole / loisir à louer,2 000 DH,Afaq,Terrain,3500 m²,NaN,NaN,À louer \n\nTerrain pour activité agricole ou ...,NaN,...,-1,NaN,2000.0,3500.0,0.0,0.0,1.0,Autre,0.571429,30.555556
4,NaN,Ferme de 4 hectares à céder,1 600 000 DH,Route D Amezmiz,Terrain,40000 m²,NaN,NaN,L'agence Stone immobilier met en location une ...,NaN,...,-1,NaN,1600000.0,40000.0,0.0,0.0,1.0,Autre,40.000000,30.555556


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 34 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       2 non-null      str    
 1   titre                    17 non-null     str    
 2   prix                     17 non-null     str    
 3   localisation             17 non-null     str    
 4   type_bien                17 non-null     str    
 5   surface                  17 non-null     str    
 6   chambres                 2 non-null      float64
 7   salles_bain              0 non-null      float64
 8   description              16 non-null     str    
 9   agence                   2 non-null      str    
 10  url                      17 non-null     str    
 11  source                   18 non-null     str    
 12  piscine                  18 non-null     int64  
 13  parking                  18 non-null     int64  
 14  ascenseur                18 non-null   

In [13]:
df.describe()

,chambres,salles_bain,piscine,parking,ascenseur,terrasse,jardin,climatisation,securite,vue,...,hammam,etage,surface_terrain,prix_num,surface_num,chambres_num,salles_bain_num,nb_pieces,prix_m2,prix_m2_median_quartier
count,2.0,0.0,18.000000,18.000000,18.0,18.0,18.000000,18.0,18.000000,18.0,...,18.0,18.0,1.0,1.100000e+01,16.000000,18.0,18.0,18.0,10.000000,14.000000
mean,0.0,NaN,0.111111,0.111111,0.0,0.0,0.222222,0.0,0.055556,0.0,...,0.0,-1.0,540.0,4.567427e+05,8475.437500,0.0,0.0,1.0,7474.756333,888.941882
std,0.0,NaN,0.323381,0.323381,0.0,0.0,0.427793,0.0,0.235702,0.0,...,0.0,0.0,NaN,7.151626e+05,12911.459076,0.0,0.0,0.0,19496.800078,2193.337966
min,0.0,NaN,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,-1.0,540.0,2.000000e+03,4.000000,0.0,0.0,1.0,0.571429,3.798077
25%,0.0,NaN,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,-1.0,540.0,5.835000e+03,487.500000,0.0,0.0,1.0,4.447115,30.555556
50%,0.0,NaN,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,-1.0,540.0,3.750000e+04,3250.000000,0.0,0.0,1.0,30.555556,30.555556
75%,0.0,NaN,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,-1.0,540.0,5.500000e+05,10000.000000,0.0,0.0,1.0,4555.454545,30.555556
max,0.0,NaN,1.000000,1.000000,0.0,0.0,1.000000,0.0,1.000000,0.0,...,0.0,-1.0,540.0,2.000000e+06,40000.000000,0.0,0.0,1.0,62500.000000,6071.428571


In [14]:
df.shape

(18, 34)

## 1. Target Protection & Deduplication
We must drop rows without a price (target) and remove duplicates.

In [15]:
if 'df' in locals():
    # 1. Drop missing target
    if 'prix_num' in df.columns:
        initial_len = len(df)
        df = df.dropna(subset=['prix_num'])
        print(f"Dropped {initial_len - len(df)} rows with missing prix_num")
    
    # 2. Drop duplicates
    initial_len = len(df)
    df = df.drop_duplicates()
    print(f"Dropped {initial_len - len(df)} duplicate rows")

Dropped 7 rows with missing prix_num
Dropped 0 duplicate rows


## 2. Feature Selection & Data Leakage Prevention
Drop non-predictive columns and columns that cause data leakage (like prix_m2).

In [16]:
if 'df' in locals():
    cols_to_drop = ['id', 'url', 'source', 'titre', 'description', 'prix', 'surface', 'localisation', 'chambres', 'salles_bain', 'prix_m2']
    existing_drops = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=existing_drops)
    print(f"Dropped columns: {existing_drops}")
    print(f"Remaining columns: {df.columns.tolist()}")

Dropped columns: ['id', 'url', 'source', 'titre', 'description', 'prix', 'surface', 'localisation', 'chambres', 'salles_bain', 'prix_m2']
Remaining columns: ['type_bien', 'agence', 'piscine', 'parking', 'ascenseur', 'terrasse', 'jardin', 'climatisation', 'securite', 'vue', 'meuble', 'neuf', 'cave', 'hammam', 'etage', 'surface_terrain', 'prix_num', 'surface_num', 'chambres_num', 'salles_bain_num', 'nb_pieces', 'quartier', 'prix_m2_median_quartier']


## 3. Intelligent Imputation
Handling missing values for features.

In [17]:
if 'df' in locals():
    # Missing values before
    print("Missing values before imputation:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    
    # 1. Numeric: Median
    if 'surface_num' in df.columns:
        df['surface_num'] = df['surface_num'].fillna(df['surface_num'].median())
    
    # 2. Rooms/Baths logic
    non_residential = ['terrain', 'locaux', 'commercial', 'bureaux']
    if 'terrain' in non_residential:
        if 'chambres_num' in df.columns: df['chambres_num'] = df['chambres_num'].fillna(0.0)
        if 'salles_bain_num' in df.columns: df['salles_bain_num'] = df['salles_bain_num'].fillna(0.0)
    else:
        if 'chambres_num' in df.columns: df['chambres_num'] = df['chambres_num'].fillna(round(df['chambres_num'].mean() if not df['chambres_num'].isnull().all() else 0))
        if 'salles_bain_num' in df.columns: df['salles_bain_num'] = df['salles_bain_num'].fillna(round(df['salles_bain_num'].mean() if not df['salles_bain_num'].isnull().all() else 0))
            
    # 3. Categorical: Mode
    cat_cols = ['agence', 'type_bien', 'quartier']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Inconnu')

    print("\nMissing values after imputation:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values before imputation:
agence                      9
surface_terrain            10
surface_num                 1
prix_m2_median_quartier     1
dtype: int64

Missing values after imputation:
surface_terrain            10
prix_m2_median_quartier     1
dtype: int64


## 4. Feature Engineering
Applying log transformations to stabilize variance.

In [18]:
if 'df' in locals():
    if 'prix_num' in df.columns:
        df['log_prix_num'] = np.log1p(df['prix_num'])
        print("Created log_prix_num")
    
    if 'surface_num' in df.columns:
        df['log_surface_num'] = np.log1p(df['surface_num'])
        print("Created log_surface_num")

Created log_prix_num
Created log_surface_num


## 5. Outlier Detection & Removal
Using 1st and 99th percentiles.

In [19]:
if 'df' in locals():
    for col in ['prix_num', 'surface_num']:
        if col in df.columns:
            q_low = df[col].quantile(0.01)
            q_hi  = df[col].quantile(0.99)
            df = df[(df[col] <= q_hi) & (df[col] >= q_low)]
            print(f"Filtered {col} between {q_low:.2f} and {q_hi:.2f}")
    
    print(f"Final shape: {df.shape}")

Filtered prix_num between 2300.00 and 1960000.00
Filtered surface_num between 14.88 and 40000.00
Final shape: (8, 25)


## 6. Exporting ML-Ready Data

In [20]:
if 'df' in locals():
    output_dir = '../../data/cleaned_data/location'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    output_path = os.path.join(output_dir, 'terrain_location_cleaned.csv')
    df.to_csv(output_path, index=False)
    print(f"Success! ML-ready data saved to {output_path}")

Success! ML-ready data saved to ../../data/cleaned_data/location/terrain_location_cleaned.csv
